In [1]:
import slangpy as spy
from pyglm import glm
import matplotlib.pyplot as plt
import numpy as np

from bvhgs import device
from bvhgs.camera import Camera
from bvhgs.gaussian import GaussianCloud
from bvhgs.renderer import Renderer
import pathlib

In [2]:
np.random.seed(348)

# Create Gaussian Buffer

In [3]:
gaussians = GaussianCloud(8)
gaussians.load_from_ply(pathlib.Path().parent / "data" / "test.ply")
len(gaussians)

16

# Load Module and Shader

In [4]:
module = device.load_module("renderer.slang")
module

SlangModule(
  name = renderer.slang,
  path = D:\Projects\bvhgs\src\bvhgs\slang\renderer.slang,
  entry_points = [
    SlangEntryPoint(name="project", stage=compute),
    SlangEntryPoint(name="cull", stage=compute),
    SlangEntryPoint(name="computeTile", stage=compute),
    SlangEntryPoint(name="buildGaussianTable", stage=compute),
    SlangEntryPoint(name="rasterize", stage=compute),
  ]
)

In [5]:
program = device.link_program([module], [])
program

ShaderProgram(
  modules = [
    SlangModule(
      name = renderer.slang,
      path = D:\Projects\bvhgs\src\bvhgs\slang\renderer.slang,
      entry_points = [
        SlangEntryPoint(name="project", stage=compute),
        SlangEntryPoint(name="cull", stage=compute),
        SlangEntryPoint(name="computeTile", stage=compute),
        SlangEntryPoint(name="buildGaussianTable", stage=compute),
        SlangEntryPoint(name="rasterize", stage=compute),
      ]
    ),
    SlangModule(
      name = sgl/device/nvapi.slang,
      path = d:\Projects\bvhgs\.venv\Lib\site-packages\slangpy\shaders\sgl/device/nvapi.slang,
      entry_points = []
    ),
  ],
  entry_points = []
)

# Projection

## Build Slang Buffer and Camera Parameter

In [6]:
# Create a buffer for the Gaussian points.
gaussian_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_3d,
    usage=spy.BufferUsage.shader_resource,
)
# Store all the gaussian points in the buffer.
gaussian_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_3d.type_layout.element_type_layout,
    gaussian_buf,
)

for i in range(len(gaussians)):
    print(gaussians[i]["position"])
    gaussian_cursor[i].write(gaussians[i])
gaussian_cursor.apply()

[-0.07346803 -0.05500152  0.20971616]
[-0.1153454  -0.06427182 -0.04809379]
[-0.20973575  0.05536801 -0.05971747]
[-0.12886406  0.16898657 -0.05390296]
[-0.22025853 -0.04800304 -0.0470501 ]
[-0.21275853  0.03373245 -0.05780813]
[ 0.45438123  0.29062247 -0.07860582]
[-0.00370298 -0.01866331 -0.12040865]
[-0.67517245 -0.05626816 -4.6401253 ]
[ 0.01646151  0.05245682 -0.10947308]
[ 0.02582444  0.28941834 -0.11321741]
[ 1.3511627  -0.34617934  1.3352257 ]
[ 0.24481386  0.04161276 -0.0915358 ]
[0.50743157 0.09866667 0.2612744 ]
[-0.07202511  0.07084413 -0.09354042]
[-0.19284827  0.09304556 -0.07198062]


In [7]:
# Create a buffer for the Gaussian2D points.
gaussian2d_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_2d,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
gaussian2d_cursor[0].read()

{'position': {0, 0, 0},
 'covariance': {{0, 0}, {0, 0}},
 'color': {0, 0, 0},
 'opacity': 0.0,
 'cachedInvCov': {{0, 0}, {0, 0}},
 'cachedDet': 0.0,
 'cachedNorm': 0.0}

In [8]:
camera = Camera(
    rotation=glm.quat(1, 0, 0, 0),
    translation=glm.vec3(0, 0, 1),
    sensor_size=glm.uvec2(512, 512),
    focal_length=64
)
camera.to_slang()

{'_rotation': [0.0, 0.0, 0.0, 1.0],
 '_translation': vec3( 0, 0, 1 ),
 '_sensorSize': uvec2( 512, 512 ),
 '_focalLength': 64}

## Dispatch Projection Kernel

In [9]:
# Create flag buffer for culling.
cull_flag_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_cull_flag,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [10]:
ker_proj = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("project")])
)
ker_proj

ComputeKernel(0x2aab8c89e60)

In [11]:
ker_proj.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_camera": camera.to_slang(),
        "g_gaussian_3d": gaussian_buf,
        "g_gaussian_2d": gaussian2d_buf,
        "g_cull_flag": cull_flag_buf
    }
)

In [12]:
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
for i in range(gaussian2d_cursor.element_count):
    print(gaussian2d_cursor[i].read()["position"])

{-0.06073162, -0.045466464, 1.2131925}
{-0.12117307, -0.06751906, 0.9610207}
{-0.22305611, 0.058884446, 0.9649797}
{-0.13620597, 0.17861442, 0.9696711}
{-0.23113339, -0.050373096, 0.9792505}
{-0.2258123, 0.035802096, 0.9665038}
{0.49314535, 0.31541604, 1.0676568}
{-0.0042098896, -0.021218156, 0.8797971}
{-675172.44, -56268.164, -3.702639}
{0.018485133, 0.058905378, 0.8922224}
{0.029121494, 0.326369, 0.9331737}
{0.57860047, -0.14824235, 2.720066}
{0.269481, 0.045805607, 0.94179225}
{0.40231657, 0.078227766, 1.3630975}
{-0.07945761, 0.07815476, 0.91207206}
{-0.20780627, 0.100262515, 0.95240116}


## Cull Gaussians

In [13]:
prefix_sum = cull_flag_buf.to_numpy().view(np.int32)
prefix_sum = np.cumsum(prefix_sum).astype(np.int32)
print(prefix_sum)

[0 0 0 0 0 0 1 1 1 2 3 3 4 5 5 5]


In [14]:
num_culled_gaussians = prefix_sum[-1]
print(num_culled_gaussians)
cull_prefix_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_cull_prefix,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
cull_prefix_buf.copy_from_numpy(prefix_sum)

5


In [15]:
# Culled gaussians
culled_gaussian_buf = device.create_buffer(
    element_count=num_culled_gaussians,
    struct_type=program.reflection.g_gaussian_2d_culled,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [16]:
ker_cull = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("cull")])
)
ker_cull.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_gaussian_2d": gaussian2d_buf,
        "g_cull_flag": cull_flag_buf,
        "g_cull_prefix": cull_prefix_buf,
        "g_gaussian_2d_culled": culled_gaussian_buf
    }
)

In [17]:
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d_culled.type_layout.element_type_layout,
    culled_gaussian_buf,
)
for i in range(gaussian2d_cursor.element_count):
    print(gaussian2d_cursor[i].read()["position"])

{0.49314535, 0.31541604, 1.0676568}
{0.018485133, 0.058905378, 0.8922224}
{0.029121494, 0.326369, 0.9331737}
{0.269481, 0.045805607, 0.94179225}
{0.40231657, 0.078227766, 1.3630975}
